You just got hired as the first and only data practitioner at a small business experiencing exponential growth. The company needs more structured processes, guidelines, and standards. Your first mission is to structure the human resources data. The data is currently scattered across teams and files and comes in various formats: Excel files, CSVs, JSON files...

You'll work with the following data in the `datasets` folder:
- __Office addresses__
    - Saved in `office_addresses.csv`. 
    - If the value for office is `NaN`, then the employee is remote.
- __Employee addresses__
    - Saved on the first tab of `employee_information.xlsx`.
- __Employee emergency contacts__ 
    - Saved on the second tab of `employee_information.xlsx`; this tab is called `emergency_contacts`. 
    - However, this sheet was edited at some point, and ***the headers were removed***! The HR manager let you know that they should be: `employee_id`, `last_name`, `first_name`, `emergency_contact`, `emergency_contact_number`, and `relationship`.
- __Employee roles, teams, and salaries__ 
    - This information has been exported from the company's human resources management system into a JSON file titled `employee_roles.json`. Here are the first few lines of that file:
```
{"A2R5H9":
  {
    "title": "CEO",
    "monthly_salary": "$4500",
    "team": "Leadership"
  },
 ...
}
```

In [107]:
import pandas as pd
office_address_df = pd.read_csv('datasets/office_addresses.csv')
employee_address_df = pd.read_excel('datasets/employee_information.xlsx')
emergency_address_df = pd.read_excel('datasets/employee_information.xlsx', sheet_name = 'emergency_contacts', header= None)
emergency_address_df.columns = ['employee_id', 'last_name', 'first_name', 'emergency_contact', 'emergency_contact_number', 'relationship']
employee_roles_df = pd.read_json('datasets/employee_roles.json', orient = 'index')
employee_roles_df.reset_index(inplace= True)
employee_roles_df.rename(columns = {'index': 'employee_id'}, inplace = True )
employee_address_df.shape

(4, 7)

In [108]:
employee_ver_1 = employee_address_df.merge(emergency_address_df, on = 'employee_id', how= 'left')
employee_ver_2 = employee_ver_1.merge(employee_roles_df, on= 'employee_id',how = 'left')
employee_ver_1.shape

(4, 12)

In [109]:
office_map = office_address_df.set_index('office_country')['office'].to_dict()
employee_ver_2['office'] = employee_ver_2['employee_country'].map(office_map)
employee_ver_2['office'].fillna('Remote', inplace=True)
employees_final = employee_ver_2.merge(office_address_df, on = 'office', how = 'left')
employees_final = employees_final.drop(columns = ['last_name', 'first_name'])
employees_final = employees_final[['employee_id', 'employee_first_name', 'employee_last_name',
       'employee_country', 'employee_city', 'employee_street',
       'employee_street_number', 'emergency_contact',
       'emergency_contact_number', 'relationship', 'monthly_salary',
       'team', 'title', 'office', 'office_country', 'office_city', 'office_street',
       'office_street_number']]
employees_final = employees_final.set_index('employee_id')
print(employees_final.columns)

Index(['employee_first_name', 'employee_last_name', 'employee_country',
       'employee_city', 'employee_street', 'employee_street_number',
       'emergency_contact', 'emergency_contact_number', 'relationship',
       'monthly_salary', 'team', 'title', 'office', 'office_country',
       'office_city', 'office_street', 'office_street_number'],
      dtype='object')


In [110]:
office_cols = employees_final.columns[employees_final.columns.str.startswith('office')]
employees_final[office_cols] = employees_final[office_cols].fillna('Remote')
print(employees_final)

            employee_first_name  ... office_street_number
employee_id                      ...                     
A2R5H9                      Jax  ...                 38.0
H8K0L6                     Tara  ...                207.0
G4R7V0                    Gemma  ...                350.0
M1Z7U9                      Tig  ...               Remote

[4 rows x 17 columns]
